# 04 - End-to-End Training (Kaggle)

Self-contained end-to-end training notebook (run it on **Kaggle**, GPU optional).

**How to run on Kaggle:**
1. New Notebook → **File → Import Notebook…** → select this file.
2. **Settings → Accelerator → GPU** (optional but recommended).
3. **+ Add Input → Kaggle Datasets** → **`misrakahmed/vegetable-image-dataset`**.
4. Run all cells.

The architecture, input size and preprocessing are identical to the model shipped
in `models/vegetable_cnn.h5` (see `03_model_architecture.ipynb` / `src/models.py`).
Outputs written to `/kaggle/working/`: best model (`.keras` + `.h5`), `class_names.json`,
training curves, confusion matrix and per-class report — mirroring the local `reports/` layout.

In [ ]:

import json
import os
import zipfile
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

KAGGLE_INPUT_BASE = Path("/kaggle/input/vegetable-image-dataset")
KAGGLE_WORKING = Path("/kaggle/working")
WORKING_FIGURES = KAGGLE_WORKING / "report" / "figures"
WORKING_METRICS = KAGGLE_WORKING / "report" / "metrics"

print(f"TensorFlow {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print("GPU ENABLED:", [g.name for g in gpus] if gpus else "none (CPU)")


In [ ]:

SEED = 42
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SPLITS = ("train", "validation", "test")

tf.random.set_seed(SEED)
np.random.seed(SEED)

def find_splits(root: Path):
    if all((root / s).is_dir() for s in SPLITS):
        return {s: root / s for s in SPLITS}
    for sub in [p for p in root.iterdir() if p.is_dir()]:
        found = find_splits(sub)
        if found:
            return found
    return None

splits = find_splits(KAGGLE_INPUT_BASE)

if not splits:
    zips = list(KAGGLE_INPUT_BASE.rglob("*.zip"))
    if zips:
        dest = KAGGLE_WORKING / "vegetable_dataset"
        dest.mkdir(parents=True, exist_ok=True)
        for z in zips:
            print(f"Extracting {z.name} -> {dest}")
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
        splits = find_splits(dest)

if not splits:
    raise SystemExit("Add the vegetable-image-dataset input (see instructions above).")

for name, path in splits.items():
    print(f"{name:10s}: {path}")


## 1) Data pipelines (raw pixels, geometric augmentation)

In [ ]:

train_ds = tf.keras.utils.image_dataset_from_directory(
    splits["train"], labels="inferred", label_mode="categorical",
    color_mode="rgb", batch_size=BATCH_SIZE, image_size=IMG_SIZE,
    shuffle=True, seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    splits["validation"], labels="inferred", label_mode="categorical",
    color_mode="rgb", batch_size=BATCH_SIZE, image_size=IMG_SIZE, shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    splits["test"], labels="inferred", label_mode="categorical",
    color_mode="rgb", batch_size=BATCH_SIZE, image_size=IMG_SIZE, shuffle=False,
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"{NUM_CLASSES} classes: {', '.join(class_names)}")
assert val_ds.class_names == class_names and test_ds.class_names == class_names


In [ ]:

AUTOTUNE = tf.data.AUTOTUNE

# Range-preserving augmentation (flip / rotate / zoom). NO pixel rescaling:
# the model expects raw [0, 255] values.
augmenter = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.15, fill_mode="nearest", seed=SEED),
        tf.keras.layers.RandomZoom(0.1, fill_mode="nearest", seed=SEED),
    ],
    name="augment",
)

def prep_aug(x, y):
    return augmenter(x, training=True), y

def prep_plain(x, y):
    return x, y

train_ds = train_ds.map(prep_aug, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(prep_plain, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_ds.map(prep_plain, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

for name, ds in (("train", train_ds), ("validation", val_ds), ("test", test_ds)):
    print(f"{name:10s}: {len(ds):4d} batches x {BATCH_SIZE} = {len(ds) * BATCH_SIZE:,} images")


## 2) Model (same as `src/models.py`)

In [ ]:

model = tf.keras.Sequential(
    [
        tf.keras.layers.Conv2D(32, (5, 5), activation="relu", input_shape=IMG_SIZE + (3,)),
        tf.keras.layers.Conv2D(32, (5, 5), activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="vegetable_cnn",
)
model.summary()


## 3) Callbacks

In [ ]:

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

BEST_MODEL_PATH = KAGGLE_WORKING / "best_vegetable_cnn.keras"
callbacks = [
    ModelCheckpoint(str(BEST_MODEL_PATH), monitor="val_accuracy", mode="max",
                    save_best_only=True, save_weights_only=False, verbose=1),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]
print(f"Best model saved to: {BEST_MODEL_PATH}")


## 4) Training

In [ ]:

EPOCHS = 30
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    callbacks=callbacks, verbose=1)


In [ ]:

WORKING_FIGURES.mkdir(parents=True, exist_ok=True)
WORKING_METRICS.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, key in zip(axes, ("accuracy", "loss")):
    ax.plot(history.history[key], "-o", ms=3, label=f"train {key}")
    ax.plot(history.history[f"val_{key}"], "-s", ms=3, label=f"val {key}")
    ax.set_title(key); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORKING_FIGURES / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()


## 5) Test-set evaluation

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc * 100:.2f}%")

y_true, y_pred = [], []
for x, y in test_ds:
    probs = model(x, training=False).numpy()
    y_true.extend(np.argmax(y.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar_kws={"shrink": 0.75},
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Test-set confusion matrix (n = {len(y_true):,})")
plt.tight_layout()
plt.savefig(WORKING_FIGURES / "confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

report = classification_report(y_true, y_pred, target_names=class_names,
                               zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).T
report_df.to_csv(WORKING_METRICS / "classification_report.csv")
with pd.option_context("display.float_format", "{:.3f}".format):
    print(report_df.to_string())


## 6) Export artefacts

In [ ]:

# Full Keras archive (also the best checkpoint) -> .h5 for the local app.
model.save(KAGGLE_WORKING / "vegetable_cnn.h5")

class_map = {str(i): name for i, name in enumerate(class_names)}
(KAGGLE_WORKING / "class_names.json").write_text(json.dumps(class_map, indent=2))

print("Files in /kaggle/working/:")
for f in sorted(KAGGLE_WORKING.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  - {f.relative_to(KAGGLE_WORKING)}  ({size_mb:.2f} MB)")


## After the run
- Download `vegetable_cnn.h5` + `class_names.json` into the repo `models/` folder.
- Optionally move `report/` contents into the local `reports/` folder.
- The GPU is only needed for training; the exported `.h5` runs fine on CPU.